### SETUP 


In [1]:
import os
import glob
import json
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np

# 1. Connexion et bridage
con = duckdb.connect()
con.execute("PRAGMA memory_limit='32GB'")
con.execute("PRAGMA threads=4")

# 2. Chargement de la table brute et du dictionnaire
DATA_DIR = Path("/data/gdelt/gdelt_parquet_db")
SOURCE_MAP_PATH = Path("/data/gdelt/gdelt_sources_mapping.json")
con.execute(f"CREATE OR REPLACE VIEW gkg AS SELECT * FROM read_parquet('{DATA_DIR / 'gdelt_*.parquet'}')")

with open(SOURCE_MAP_PATH, "r", encoding="utf-8") as f:
    source_map = json.load(f)
src_df = pd.DataFrame({
    "SourceCommonName_ID": [int(k) for k in source_map["id_to_source"].keys()],
    "SourceCommonName": list(source_map["id_to_source"].values()),
})
con.register("src_map", src_df)

print("⏳ Création de la Mega-Vue Optimisée (Filtres WordCount et Thèmes en premier)...")

con.execute(r"""
    CREATE OR REPLACE VIEW gkg_clean AS
    WITH raw_filtered AS (
        SELECT * FROM gkg
        WHERE regexp_matches(CAST(DATE AS VARCHAR), '^\d{14}$')
          AND GKGRECORDID != '20210925181500-T1111'
          AND EnhancedThemes IS NOT NULL AND EnhancedThemes != ''
          AND CAST(WordCount AS INTEGER) BETWEEN 150 AND 5500 
    )
    SELECT 
        CASE 
            WHEN COALESCE(r.SourceCommonName_ID, 0) = 0 THEN m.SourceCommonName_ID
            ELSE r.SourceCommonName_ID
        END AS SourceCommonName_ID,
        r.DATE, r.WordCount, r.EnhancedThemes
    FROM raw_filtered r
    LEFT JOIN src_map m 
      ON RTRIM(regexp_extract(r.DocumentIdentifier, 'https?://(?:www\.)?([^/?:]+)', 1), '.') = m.SourceCommonName
    WHERE COALESCE(r.SourceCommonName_ID, 0) != 0 OR m.SourceCommonName_ID IS NOT NULL;
""")
print("✔ Vue 'gkg_clean' créée avec succès !")

⏳ Création de la Mega-Vue Optimisée (Filtres WordCount et Thèmes en premier)...
✔ Vue 'gkg_clean' créée avec succès !


### Features 

In [3]:
print("⏳ [1/4] Calcul des statistiques de base (Volume, Wordcount, Thèmes/art)...")
con.execute("""
    CREATE OR REPLACE TABLE base_stats AS
    SELECT 
        SourceCommonName_ID,
        COUNT(*) AS total_articles,
        LN(1 + COUNT(*)) AS log_total_articles,
        AVG(WordCount) AS mean_wordcount,
        AVG(CASE 
            WHEN EnhancedThemes IS NULL OR EnhancedThemes = '' THEN 0 
            ELSE len(list_distinct(list_transform(string_split(EnhancedThemes, ';'), x -> split_part(x, ',', 1))))
        END) AS mean_themes_per_art
    FROM gkg_clean
    GROUP BY SourceCommonName_ID;
""")

print("⏳ [2/4] Calcul de la longévité (Years active)...")
con.execute("""
    CREATE OR REPLACE TABLE temporal_stats AS
    SELECT 
        SourceCommonName_ID,
        COUNT(DISTINCT CAST(SUBSTRING(CAST(DATE AS VARCHAR), 1, 4) AS INTEGER)) AS years_active
    FROM gkg_clean
    GROUP BY SourceCommonName_ID;
""")

print("⏳ [3/4] Extraction des thèmes et calcul de l'entropie thématique globale...")
con.execute("""
    -- Comptage des thèmes (table intermédiaire réutilisable)
    CREATE OR REPLACE TABLE theme_counts AS
    SELECT 
        SourceCommonName_ID,
        split_part(raw_theme, ',', 1) AS theme,
        COUNT(*) as freq
    FROM (
        SELECT SourceCommonName_ID, unnest(string_split(EnhancedThemes, ';')) AS raw_theme
        FROM gkg_clean
    )
    WHERE raw_theme != ''
    GROUP BY 1, 2;
""")

con.execute("""
    -- Entropie globale
    CREATE OR REPLACE TABLE theme_entropy AS
    WITH source_totals AS (
        SELECT SourceCommonName_ID, SUM(freq) as total_freq
        FROM theme_counts
        GROUP BY 1
    ),
    global_max AS (
        SELECT COUNT(DISTINCT theme) as T_global FROM theme_counts
    )
    SELECT 
        tc.SourceCommonName_ID,
        SUM(- (tc.freq * 1.0 / st.total_freq) * ln(tc.freq * 1.0 / st.total_freq)) / ln(NULLIF((SELECT T_global FROM global_max), 1)) AS h_norm_themes
    FROM theme_counts tc
    JOIN source_totals st ON tc.SourceCommonName_ID = st.SourceCommonName_ID
    GROUP BY tc.SourceCommonName_ID;
""")

print("⏳ [4/4] Calcul des métriques économiques (Ratio et Entropie d'éco)...")
con.execute("""
    -- Ratio Economique (utilisant la table theme_counts déjà découpée !)
    CREATE OR REPLACE TABLE eco_ratios AS
    SELECT 
        SourceCommonName_ID,
        SUM(CASE WHEN theme LIKE '%ECON%' THEN freq ELSE 0 END)::DOUBLE / NULLIF(SUM(freq), 0) AS ratio_eco
    FROM theme_counts
    GROUP BY SourceCommonName_ID;
""")

con.execute("""
    -- Sous-table des thèmes uniquement économiques
    CREATE OR REPLACE TABLE eco_theme_counts AS
    SELECT * FROM theme_counts WHERE theme LIKE '%ECON%';
""")

con.execute("""
    -- Entropie économique
    CREATE OR REPLACE TABLE eco_entropy AS
    WITH source_totals AS (
        SELECT SourceCommonName_ID, SUM(freq) as total_freq
        FROM eco_theme_counts
        GROUP BY 1
    ),
    global_max AS (
        SELECT COUNT(DISTINCT theme) as T_global FROM eco_theme_counts
    )
    SELECT 
        tc.SourceCommonName_ID,
        SUM(- (tc.freq * 1.0 / st.total_freq) * ln(tc.freq * 1.0 / st.total_freq)) / NULLIF(ln(NULLIF((SELECT T_global FROM global_max), 1)), 0) AS h_norm_eco
    FROM eco_theme_counts tc
    JOIN source_totals st ON tc.SourceCommonName_ID = st.SourceCommonName_ID
    GROUP BY tc.SourceCommonName_ID;
""")

print("⏳ Assemblage des features unifiées...")

df_features_v2 = con.execute("""
    SELECT 
        b.SourceCommonName_ID,
        t.years_active,
        b.log_total_articles,
        b.mean_wordcount,
        COALESCE(e.h_norm_themes, 0) AS h_norm_themes,
        COALESCE(eco.ratio_eco, 0) AS ratio_eco,
        COALESCE(eco_ent.h_norm_eco, 0) AS h_norm_eco
        -- (Ajoute ici tes autres features CV, etc.)
    FROM base_stats b
    JOIN temporal_stats t ON b.SourceCommonName_ID = t.SourceCommonName_ID
    LEFT JOIN theme_entropy e ON b.SourceCommonName_ID = e.SourceCommonName_ID
    LEFT JOIN eco_ratios eco ON b.SourceCommonName_ID = eco.SourceCommonName_ID
    LEFT JOIN eco_entropy eco_ent ON b.SourceCommonName_ID = eco_ent.SourceCommonName_ID
""").df()

# Calcul de log_total_articles_year
total_articles = np.expm1(df_features_v2["log_total_articles"]).round()
df_features_v2["log_total_articles_year"] = np.log1p(
    total_articles / df_features_v2["years_active"].replace(0, np.nan)
).fillna(0.0)

print(f"✔ Features calculées pour {len(df_features_v2)} sources.")

⏳ [1/4] Calcul des statistiques de base (Volume, Wordcount, Thèmes/art)...
⏳ [2/4] Calcul de la longévité (Years active)...
⏳ [3/4] Extraction des thèmes et calcul de l'entropie thématique globale...


OutOfMemoryException: Out of Memory Error: failed to pin block of size 256.0 KiB (29.8 GiB/29.8 GiB used)

Possible solutions:
* Reducing the number of threads (SET threads=X)
* Disabling insertion-order preservation (SET preserve_insertion_order=false)
* Increasing the memory limit (SET memory_limit='...GB')

See also https://duckdb.org/docs/stable/guides/performance/how_to_tune_workloads

In [4]:
print("💾 Sauvegarde de l'étape 1 et 2 en Parquet...")
con.execute("COPY base_stats TO 'base_stats_v2.parquet' (FORMAT PARQUET);")
con.execute("COPY temporal_stats TO 'temporal_stats_v2.parquet' (FORMAT PARQUET);")

print("🧹 Nettoyage de la RAM...")
# On supprime les tables de la mémoire de DuckDB. 
# Pas d'inquiétude, elles sont bien au chaud dans les fichiers Parquet !
con.execute("DROP TABLE base_stats;")
con.execute("DROP TABLE temporal_stats;")
print("✔ RAM libérée !")

💾 Sauvegarde de l'étape 1 et 2 en Parquet...
🧹 Nettoyage de la RAM...
✔ RAM libérée !


In [6]:
import gc

print("🔧 Réduction des threads pour soulager la mémoire par cœur...")
con.execute("PRAGMA threads=2")

print("⏳ [3/4] Extraction des thèmes par LOTS (Chunking) pour préserver la RAM...")

# On divise le travail en 4 lots basés sur l'ID de la source (Modulo 4)
for i in range(4):
    print(f"   -> Traitement du lot {i+1}/4...")
    con.execute(f"""
        COPY (
            SELECT 
                SourceCommonName_ID,
                split_part(raw_theme, ',', 1) AS theme,
                COUNT(*) as freq
            FROM (
                SELECT SourceCommonName_ID, unnest(string_split(EnhancedThemes, ';')) AS raw_theme
                FROM gkg_clean
                -- LE FILTRE EST ICI : on ne prend qu'un quart des sources à la fois
                WHERE CAST(SourceCommonName_ID AS BIGINT) % 4 = {i}
            )
            WHERE raw_theme != ''
            GROUP BY 1, 2
        ) TO 'theme_counts_v2_part{i}.parquet' (FORMAT PARQUET);
    """)
    # On force le nettoyage de la RAM Python/DuckDB entre chaque lot
    gc.collect() 

print("✔ Tous les lots sont extraits et sauvegardés sur le disque !")

# On unifie les 4 fichiers dans une seule VUE (coût en RAM : 0 Go)
con.execute("CREATE OR REPLACE VIEW theme_counts AS SELECT * FROM read_parquet('theme_counts_v2_part*.parquet');")

🔧 Réduction des threads pour soulager la mémoire par cœur...
⏳ [3/4] Extraction des thèmes par LOTS (Chunking) pour préserver la RAM...
   -> Traitement du lot 1/4...
   -> Traitement du lot 2/4...
   -> Traitement du lot 3/4...
   -> Traitement du lot 4/4...
✔ Tous les lots sont extraits et sauvegardés sur le disque !


In [7]:
print("⏳ Calcul de l'entropie thématique globale...")
con.execute("""
    CREATE OR REPLACE TABLE theme_entropy AS
    WITH source_totals AS (
        SELECT SourceCommonName_ID, SUM(freq) as total_freq
        FROM theme_counts
        GROUP BY 1
    ),
    global_max AS (
        SELECT COUNT(DISTINCT theme) as T_global FROM theme_counts
    )
    SELECT 
        tc.SourceCommonName_ID,
        SUM(- (tc.freq * 1.0 / st.total_freq) * ln(tc.freq * 1.0 / st.total_freq)) / ln(NULLIF((SELECT T_global FROM global_max), 1)) AS h_norm_themes
    FROM theme_counts tc
    JOIN source_totals st ON tc.SourceCommonName_ID = st.SourceCommonName_ID
    GROUP BY tc.SourceCommonName_ID;
""")

print("⏳ [4/4] Calcul des métriques économiques (Ratio et Entropie d'éco)...")

con.execute("""
    CREATE OR REPLACE TABLE eco_ratios AS
    SELECT 
        SourceCommonName_ID,
        SUM(CASE WHEN theme LIKE '%ECON%' THEN freq ELSE 0 END)::DOUBLE / NULLIF(SUM(freq), 0) AS ratio_eco
    FROM theme_counts
    GROUP BY SourceCommonName_ID;
""")

con.execute("""
    CREATE OR REPLACE TABLE eco_theme_counts AS
    SELECT * FROM theme_counts WHERE theme LIKE '%ECON%';
""")

con.execute("""
    CREATE OR REPLACE TABLE eco_entropy AS
    WITH source_totals AS (
        SELECT SourceCommonName_ID, SUM(freq) as total_freq
        FROM eco_theme_counts
        GROUP BY 1
    ),
    global_max AS (
        SELECT COUNT(DISTINCT theme) as T_global FROM eco_theme_counts
    )
    SELECT 
        tc.SourceCommonName_ID,
        SUM(- (tc.freq * 1.0 / st.total_freq) * ln(tc.freq * 1.0 / st.total_freq)) / NULLIF(ln(NULLIF((SELECT T_global FROM global_max), 1)), 0) AS h_norm_eco
    FROM eco_theme_counts tc
    JOIN source_totals st ON tc.SourceCommonName_ID = st.SourceCommonName_ID
    GROUP BY tc.SourceCommonName_ID;
""")

con.execute("DROP TABLE eco_theme_counts;")

⏳ Calcul de l'entropie thématique globale...
⏳ [4/4] Calcul des métriques économiques (Ratio et Entropie d'éco)...


In [ ]:
print("⏳ Assemblage final des features...")

df_features_v2 = con.execute("""
    SELECT 
        b.SourceCommonName_ID,
        t.years_active,
        b.log_total_articles,
        b.mean_wordcount,
        b.mean_themes_per_art,
        COALESCE(e.h_norm_themes, 0) AS h_norm_themes,
        COALESCE(eco.ratio_eco, 0) AS ratio_eco,
        COALESCE(eco_ent.h_norm_eco, 0) AS h_norm_eco
    FROM read_parquet('base_stats_v2.parquet') b
    JOIN read_parquet('temporal_stats_v2.parquet') t ON b.SourceCommonName_ID = t.SourceCommonName_ID
    LEFT JOIN theme_entropy e ON b.SourceCommonName_ID = e.SourceCommonName_ID
    LEFT JOIN eco_ratios eco ON b.SourceCommonName_ID = eco.SourceCommonName_ID
    LEFT JOIN eco_entropy eco_ent ON b.SourceCommonName_ID = eco_ent.SourceCommonName_ID
""").df()

# # Sauvegarde globale finale pour avoir la base V2
# df_features_v2.to_parquet("features_sources_v2_FINAL.parquet", index=False)
# print("✔ Assemblage terminé et sauvegardé dans 'features_sources_v2_FINAL.parquet' !")

⏳ Assemblage final des features...
✔ Assemblage terminé et sauvegardé dans 'features_sources_v2_FINAL.parquet' !


In [9]:
import numpy as np

print("⏳ Calcul de 'log_total_articles_year'...")
# 1. On retrouve le nombre total d'articles exact
total_articles = np.expm1(df_features_v2["log_total_articles"]).round()

# 2. Calcul du log de la moyenne par année active
df_features_v2["log_total_articles_year"] = np.log1p(
    total_articles / df_features_v2["years_active"].replace(0, np.nan)
).fillna(0.0)

print("⏳ Génération de la nouvelle sélection Gold Standard (V2)...")

# Application stricte de ta méthodologie (Étape 4 de ton papier)
cond_structure_v2 = (df_features_v2["years_active"] >= 4.0) & (df_features_v2["log_total_articles_year"] >= 6.5)
cond_generaliste_v2 = (df_features_v2["h_norm_themes"] >= 0.50) & (df_features_v2["ratio_eco"] >= 0.02)
cond_eco_niche_v2 = (df_features_v2["ratio_eco"] >= 0.10) & (df_features_v2["h_norm_eco"] >= 0.35)

df_whitelist_v2 = df_features_v2[cond_structure_v2 & (cond_generaliste_v2 | cond_eco_niche_v2)].copy()

set_v2 = set(df_whitelist_v2["SourceCommonName_ID"])
print(f"✔ Nombre de sources retenues (V2) : {len(set_v2):,}")

⏳ Calcul de 'log_total_articles_year'...
⏳ Génération de la nouvelle sélection Gold Standard (V2)...
✔ Nombre de sources retenues (V2) : 12,721


### RECORD

In [14]:
print("⏳ Préparation des vues pour les statistiques de filtrage...")

# Étape 1 : Dates valides et IDs réparés
con.execute("""
    CREATE OR REPLACE VIEW gkg_s1 AS 
    WITH raw_filtered AS (
        SELECT * FROM gkg 
        WHERE regexp_matches(CAST(DATE AS VARCHAR), '^\d{14}$') 
          AND GKGRECORDID != '20210925181500-T1111'
    )
    SELECT r.* EXCLUDE (SourceCommonName_ID),
           CASE WHEN COALESCE(r.SourceCommonName_ID, 0) = 0 THEN m.SourceCommonName_ID ELSE r.SourceCommonName_ID END AS SourceCommonName_ID
    FROM raw_filtered r
    LEFT JOIN src_map m ON RTRIM(regexp_extract(r.DocumentIdentifier, 'https?://(?:www\.)?([^/?:]+)', 1), '.') = m.SourceCommonName
    WHERE COALESCE(r.SourceCommonName_ID, 0) != 0 OR m.SourceCommonName_ID IS NOT NULL;
""")

# Étape 2 : + Thèmes présents
con.execute("CREATE OR REPLACE VIEW gkg_s2 AS SELECT * FROM gkg_s1 WHERE EnhancedThemes IS NOT NULL AND EnhancedThemes != '';")

# Étape 3 : + WordCount valide (150 - 5500)
con.execute("CREATE OR REPLACE VIEW gkg_s3 AS SELECT * FROM gkg_s2 WHERE CAST(WordCount AS INTEGER) BETWEEN 150 AND 5500;")

# Étape 4 : + Longévité >= 2 ans
con.execute("""
    CREATE OR REPLACE TABLE valid_sources_s4 AS 
    SELECT SourceCommonName_ID FROM gkg_s3 
    GROUP BY SourceCommonName_ID 
    HAVING COUNT(DISTINCT SUBSTRING(CAST(DATE AS VARCHAR), 1, 4)) >= 2;
""")
con.execute("CREATE OR REPLACE VIEW gkg_s4 AS SELECT a.* FROM gkg_s3 a INNER JOIN valid_sources_s4 b ON a.SourceCommonName_ID = b.SourceCommonName_ID;")

# Étape 5 : Le Gold Standard (Filtré avec ta Whitelist Pandas)
con.register("whitelist_ids", df_whitelist_v2[['SourceCommonName_ID']])
con.execute("CREATE OR REPLACE VIEW gkg_s5 AS SELECT a.* FROM gkg_s4 a INNER JOIN whitelist_ids b ON a.SourceCommonName_ID = b.SourceCommonName_ID;")

print("✔ Les 5 étapes sont prêtes à être mesurées !")

<>:4: SyntaxWarning: invalid escape sequence '\d'
<>:4: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_1284004/4034705953.py:4: SyntaxWarning: invalid escape sequence '\d'
  con.execute("""


⏳ Préparation des vues pour les statistiques de filtrage...
✔ Les 5 étapes sont prêtes à être mesurées !


In [ ]:
import pandas as pd

print("⏳ Création de la table de synthèse (Un seul scan rapide des données)...")

# 1. On crée une table temporaire ultra-légère (Sources, Année, Nombre d'articles)
# L'astuce mathématique (DATE / 10000000000) extrait l'année 100x plus vite qu'un SUBSTRING textuel.
con.execute("""
    CREATE TEMPORARY TABLE IF NOT EXISTS src_yr_summary AS
    SELECT 
        SourceCommonName_ID,
        CAST(CAST(DATE AS BIGINT) / 10000000000 AS INTEGER) AS yr,
        COUNT(*) as articles
    FROM gkg_clean
    GROUP BY 1, 2;
""")


In [20]:
import pandas as pd

print("⏳ Création de la table de synthèse (Un seul scan rapide des données)...")

# 1. On crée la table temporaire ultra-légère si elle n'existe pas encore
con.execute("""
    CREATE TEMPORARY TABLE IF NOT EXISTS src_yr_summary AS
    SELECT 
        SourceCommonName_ID,
        CAST(CAST(DATE AS BIGINT) / 10000000000 AS INTEGER) AS yr,
        COUNT(*) as articles
    FROM gkg_clean
    GROUP BY 1, 2;
""")

# 2. CORRECTION DU CATALOGUE : On détruit toute ancienne vue ou table portant ce nom
con.execute("DROP VIEW IF EXISTS whitelist_ids;")
con.execute("DROP TABLE IF EXISTS whitelist_ids;")

# Puis on crée la table proprement
con.execute("""
    CREATE TEMPORARY TABLE whitelist_ids AS
    SELECT column0::BIGINT AS SourceCommonName_ID 
    FROM read_csv_auto('gold_standard_whitelist_v2.txt', header=False);
""")

print("✔ Tables créées ! Calcul instantané des statistiques...")

# 3. Les statistiques des étapes 1 et 2 ("en dur")
table1_data = [
    {'Stage': '(1)', 'Sources': 300237, 'Articles': 1580880862, 'Avg. art. per source': 5265.4, 'Avg. art. per yr/src': 1966.4, 'Avg. active src/yr': 66994.9},
    {'Stage': '(2)', 'Sources': 285662, 'Articles': 1434361216, 'Avg. art. per source': 5021.2, 'Avg. art. per yr/src': 1852.7, 'Avg. active src/yr': 64515.8}
]

table2_data = {
    1: [164664, 155603],
    2: [50568, 48081],
    3: [19972, 19014],
    4: [15082, 14419],
    5: [9162, 8770],
    6: [6875, 6588],
    7: [5524, 5324],
    8: [4850, 4730],
    9: [4920, 4740],
    10: [4400, 4333],
    11: [5553, 5489],
    12: [8667, 8571]
}

# 4. Les requêtes pour les étapes 3, 4 et 5
stages = [
    ('(3)', "SELECT * FROM src_yr_summary"),
    ('(4)', """
        WITH valid_sources AS (
            SELECT SourceCommonName_ID FROM src_yr_summary GROUP BY 1 HAVING COUNT(yr) >= 2
        )
        SELECT s.* FROM src_yr_summary s JOIN valid_sources v ON s.SourceCommonName_ID = v.SourceCommonName_ID
    """),
    ('(5)', """
        SELECT s.* FROM src_yr_summary s JOIN whitelist_ids w ON s.SourceCommonName_ID = w.SourceCommonName_ID
    """)
]

for stage_name, base_query in stages:
    print(f"   -> Analyse de l'étape {stage_name}...")
    
    # --- TABLE 1 ---
    res = con.execute(f"""
        WITH base_data AS ({base_query}),
        source_years AS (
            SELECT SourceCommonName_ID, COUNT(yr) as ya
            FROM base_data
            GROUP BY SourceCommonName_ID
        )
        SELECT 
            COALESCE((SELECT SUM(articles) FROM base_data), 0) as articles,
            COUNT(*) as sources,
            SUM(ya) as total_src_years,
            (SELECT COUNT(DISTINCT yr) FROM base_data) as total_years
        FROM source_years;
    """).fetchone()
    
    articles, sources, total_src_years, total_years = res
    
    avg_art_src = articles / sources if sources else 0
    avg_art_yr_src = articles / total_src_years if total_src_years else 0
    avg_active_src_yr = total_src_years / total_years if total_years else 0
    
    table1_data.append({
        'Stage': stage_name,
        'Sources': sources,
        'Articles': int(articles),
        'Avg. art. per source': avg_art_src,
        'Avg. art. per yr/src': avg_art_yr_src,
        'Avg. active src/yr': avg_active_src_yr
    })
    
    # --- TABLE 2 ---
    dist = con.execute(f"""
        WITH base_data AS ({base_query}),
        source_years AS (
            SELECT SourceCommonName_ID, COUNT(yr) as ya
            FROM base_data
            GROUP BY SourceCommonName_ID
        )
        SELECT ya, COUNT(*) FROM source_years GROUP BY ya;
    """).fetchall()
    
    dist_dict = {row[0]: row[1] for row in dist}
    for year in range(1, 13):
        table2_data[year].append(dist_dict.get(year, 0))

# 5. Affichage propre
df_table1 = pd.DataFrame(table1_data).set_index('Stage')

df_table2 = pd.DataFrame(table2_data).T
df_table2.columns = [f'({i})' for i in range(1, 6)]
df_table2.index.name = 'Years of activity'
df_table2.index = [f'{y} year{"s" if y>1 else ""}' for y in df_table2.index]

print("\n================================================================================")
print("TABLE 1 : GKG statistics by filtering version (Prêt pour LaTeX)")
print("================================================================================")
display(df_table1.style.format("{:,.1f}", subset=['Avg. art. per source', 'Avg. art. per yr/src', 'Avg. active src/yr']).format("{:,}", subset=['Sources', 'Articles']))

print("\n================================================================================")
print("TABLE 2 : Distribution of sources by duration of activity (Prêt pour LaTeX)")
print("================================================================================")
display(df_table2.style.format("{:,}"))

⏳ Création de la table de synthèse (Un seul scan rapide des données)...
✔ Tables créées ! Calcul instantané des statistiques...
   -> Analyse de l'étape (3)...
   -> Analyse de l'étape (4)...
   -> Analyse de l'étape (5)...

TABLE 1 : GKG statistics by filtering version (Prêt pour LaTeX)


,Sources,Articles,Avg. art. per source,Avg. art. per yr/src,Avg. active src/yr
Stage,,,,,
(1),"300,237","1,580,880,862","5,265.4","1,966.4","66,994.9"
(2),"285,662","1,434,361,216","5,021.2","1,852.7","64,515.8"
(3),"248,561","1,103,417,450","4,439.2","1,559.2","58,973.3"
(4),"119,045","1,100,558,063","9,244.9","1,903.5","48,180.3"
(5),"12,721","910,728,514","71,592.5","7,290.3","10,410.3"



TABLE 2 : Distribution of sources by duration of activity (Prêt pour LaTeX)


,(1),(2),(3),(4),(5)
1 year,"164,664","155,603","129,516",0,0
2 years,"50,568","48,081","42,063","42,063",0
3 years,"19,972","19,014","17,385","17,385",0
4 years,"15,082","14,419","13,356","13,356",597
5 years,"9,162","8,770","8,299","8,299",505
6 years,"6,875","6,588","6,336","6,336",620
7 years,"5,524","5,324","5,046","5,046",772
8 years,"4,850","4,730","4,450","4,450",858
9 years,"4,920","4,740","4,569","4,569","1,182"
10 years,"4,400","4,333","4,217","4,217","1,400"


In [17]:
print("💾 Sauvegarde de la Whitelist en format TXT (Rétrocompatibilité)...")

# On extrait les identifiants uniques de la V2
retained_ids_list = df_whitelist_v2['SourceCommonName_ID'].unique().tolist()

# On écrit le fichier texte, un ID par ligne, comme dans la V1
with open("gold_standard_whitelist_v2.txt", "w", encoding="utf-8") as f:
    for src_id in sorted(retained_ids_list):
        f.write(f"{src_id}\n")

print(f"✔ {len(retained_ids_list):,} IDs sauvegardés dans 'gold_standard_whitelist_v2.txt' !")

# (Optionnel) Tu peux toujours sauvegarder l'intégralité des features en parquet à côté pour tes archives
df_features_v2.to_parquet("all_sources_features_v2.parquet", index=False)

💾 Sauvegarde de la Whitelist en format TXT (Rétrocompatibilité)...
✔ 12,721 IDs sauvegardés dans 'gold_standard_whitelist_v2.txt' !


In [ ]:
print("💾 SAUVEGARDE FINALE DES DONNÉES...")

# 1. Le référentiel complet de toutes les sources (avec toutes les colonnes)
df_features_v2.to_parquet("all_sources_features_v2.parquet", index=False)
print("✔ 'all_sources_features_v2.parquet' sauvegardé !")

# 2. La Whitelist simple pour DuckDB (format TXT)
retained_ids_list = df_whitelist_v2['SourceCommonName_ID'].unique().tolist()
with open("gold_standard_whitelist_v2.txt", "w", encoding="utf-8") as f:
    for src_id in sorted(retained_ids_list):
        f.write(f"{src_id}\n")
print(f"✔ {len(retained_ids_list):,} IDs sauvegardés dans 'gold_standard_whitelist_v2.txt' !")